# Annotate VEP variant table with consequence from the VAT
A preprocessing step

In [2]:
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

In [3]:
input_vep_file = 'variant_tables/IGVFFI1443TQDN.csv.gz'
gene_vat_dir = '../vat'
gene_cohort_dir = 'gene_cohorts'
#intermediate_dir = 'intermediates/per-gene-veps'
output_vep_file = 'variant_tables/annotated_filtered_VEP_calibrations.tsv.gz'

In [4]:
# Load VEP table
vep_df = pd.read_csv(input_vep_file, dtype=str)


In [5]:
# Filter to genes with cohorts
cohort_genes = {gene for gene in tqdm(vep_df['gene_symbol'].drop_duplicates()) if Path(gene_cohort_dir, f'{gene} cohort.parquet').exists()}
display(cohort_genes)

In [13]:
filtered_vep_df = vep_df[
    vep_df['gene_symbol'].isin(cohort_genes)
].assign(
    POS = lambda df: pd.to_numeric(df['POS'], downcast='integer')
)

In [14]:
# Check VAT table files
for gene in filtered_vep_df['gene_symbol'].drop_duplicates():
    if not Path(gene_vat_dir, f"{gene}_vat.tsv").exists():
        print(f'VAT for {gene} not found')


In [25]:
annotated_dfs = {}
for gene, gene_df in tqdm(filtered_vep_df.groupby('gene_symbol')):
    gene_vat = pd.read_table(
        Path(gene_vat_dir, f"{gene}_vat.tsv"),
        dtype=str
    ).assign(pos = lambda df: pd.to_numeric(df['position'], downcast='integer'))
    
    annotated_dfs[gene] = gene_df.merge(
        gene_vat,
        how = 'left',
        left_on = ['#CHROM', 'POS', 'REF', 'ALT'],
        right_on = ['contig', 'pos', 'ref_allele', 'alt_allele'],
        suffixes = (None, '_VAT')
    )

In [26]:
annotated_df = pd.concat(annotated_dfs, names=['Gene'])

In [27]:
annotated_df[['gene_symbol', 'consequence']].value_counts(dropna=False).sort_index()

In [28]:
annotated_df.to_csv(output_vep_file, sep='\t')